In [126]:
import pandas as pd
import json

In [127]:
# load fgo json data
try:
    # 2. Open the file using the 'with' statement
    with open('./fantasyGO_data.json', 'r') as f:
        # 3. Load the JSON data from the file
        fgo_23_24 = json.load(f)

    # The 'data' variable now holds your JSON content
    # In Jupyter, placing the variable at the end of the cell will display it
    print("✅ JSON file loaded successfully!")

except FileNotFoundError:
    print(f"❌ Error: The file ./fantasyGO_data.json was not found. Please check the path.")
except json.JSONDecodeError:
    print(f"❌ Error: The file ./fantasyGO_data.json is not a valid JSON file. Please check its contents.")

✅ JSON file loaded successfully!


In [ ]:
# 2. Flatten the nested fgo_23_24
all_picks = []

# Loop through each league in the top-level list
for gameweek_data in fgo_23_24:
    gameweek_info = {
        'gameweek': gameweek_data.get('contest').split(' ')[1],
        'contestant_no': gameweek_data.get('contestant_no'),
        'prize_pool': gameweek_data.get('prize_pool')
    }
    # Loop through each page within the league (e.g., "page_1", "page_2")
    for i in range(1, 100): # Assuming a max of 99 pages
        page_key = f'page_{i}'
        if page_key in gameweek_data:
            # Loop through each manager's entry on the page
            for entry_data in gameweek_data[page_key]:
                entry_info = {
                    'manager': entry_data.get('manager'),
                    'entry_num': entry_data.get('entry'),
                    'total_points': entry_data.get('points'),
                    'prize': entry_data.get('prize')
                }

                # Loop through each player pick in the entry
                for pick_data in entry_data.get('pick', []):
                    # Combine all the info into a single record
                    full_record = {
                        **gameweek_info,
                        **entry_info,
                        **pick_data
                    }
                    all_picks.append(full_record)
        else:
            break # Stop if the next page doesn't exist

# 3. Create the DataFrame
fgo_df = pd.DataFrame(all_picks)

# Optional: Clean up the 'points' and 'total_points' columns
fgo_df['points'] = pd.to_numeric(fgo_df['points'])
fgo_df['total_points'] = fgo_df['total_points'].str.replace(' Points', '', regex=False).astype(float)


# --- Display the Result ---
print("✅ DataFrame created successfully!")
print(f"Total rows: {len(fgo_df)}")
print("\n--- Sample of the final DataFrame ---")


fgo_df

✅ DataFrame created successfully!
Total rows: 110088

--- Sample of the final DataFrame ---


,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice
0,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Sanchez,2.0,,False,False
1,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Veltman,1.0,,False,False
2,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Wan-Bissaka,12.0,,False,False
3,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Estupiñan,7.0,,False,False
4,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Mitoma,5.0,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
110083,38,256,"R 9,500.00",MJ23,2,26.0,,B.Fernandes,0.0,,True,False
110084,38,256,"R 9,500.00",MJ23,2,26.0,,Bailey,0.0,,False,False
110085,38,256,"R 9,500.00",MJ23,2,26.0,,Haaland,3.0,,False,True
110086,38,256,"R 9,500.00",MJ23,2,26.0,,Darwin,1.0,,False,False


In [142]:
fpl_players_df = pd.read_csv('../with new features/data/vaastav/data/2023-24/players_raw.csv')
fgo_names = fgo_df['player_name'].unique().tolist()
fpl_web_names = fpl_players_df['web_name'].unique().tolist()

In [130]:
fpl_cleaned = pd.read_csv('../with new features/data/joint/23-24/merged_player_data.csv')
fpl_cleaned.columns

Index(['assists_x', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element',
       'expected_assists', 'expected_goal_involvements', 'expected_goals',
       'expected_goals_conceded',
       ...
       'npxG_5', 'xGChain_5', 'xGBuildup_5', 'xP_5', 'ownership_change',
       'percenatge_net_transfers', 'pts_bps', 'whh', 'whd', 'wha'],
      dtype='object', length=175)

In [131]:
# confirm names in fgo match those in fpl_web_names
match_df =[]
for name in fgo_names:
    if name in fpl_web_names:
        match_df.append({'name': name, 'has_match': True})
    else:
        match_df.append({'name': name, 'has_match': False})
match_df = pd.DataFrame(match_df)

match_df[~match_df['has_match']]['name'].to_list()

['Vinicius', 'Mitooma', 'Bradely', 'N.Semendo', 'De Bryune', 'Isak.']

In [132]:
fgo_missing_names_dict = {
    'Vinicius': 'Vinícius' ,
    'Mitooma': 'Mitoma',
    'Bradely': 'Bradley',
    'N.Semendo': 'N.Semedo',
    'De Bryune': 'De Bruyne',
    'Isak.': 'Isak'
}

# fpl_players_df.set_index('web_name')['fpl_name']

In [143]:
# Use .map() to look up corrected names and .fillna() to keep original names that weren't in the dict.
fgo_df['player_name'] = fgo_df['player_name'].map(fgo_missing_names_dict).fillna(fgo_df['player_name'])

# Map: web_name -> element_type
position_map = fpl_players_df.set_index('web_name')['element_type'].to_dict()

# Map: web_name -> id
id_map = fpl_players_df.set_index('web_name')['id'].to_dict()

# This looks up each corrected player_name in your new dictionaries.
fgo_df['position'] = fgo_df['player_name'].map(position_map)
fgo_df['fpl_id'] = fgo_df['player_name'].map(id_map)

fgo_df.loc[fgo_df['fpl_id'] == 862, 'fpl_id'] = 204

In [154]:
# 1. Calculate the number of managers for each gameweek
# This calculates the size of each group and divides by 11
manager_counts = fgo_df.groupby('gameweek')['gameweek'].transform('size') / 11
fgo_df['managers'] = manager_counts

# 2. Calculate the ownership counts for each player within each gameweek
ownership_counts = fgo_df.groupby(['gameweek', 'player_name'])['player_name'].transform('size')
fgo_df['owned_by'] = ownership_counts

# 3. Calculate the ownership percentage in one go
# We need to get the manager count for each group to divide by
# manager_counts_for_calc = fgo_df.groupby(['gameweek', 'player_name'])['managers'].first()
ownership_percentage = round((fgo_df['owned_by']  / fgo_df['managers']) * 100, 2)
fgo_df['ownership_percent'] = ownership_percentage

fgo_df[fgo_df['fpl_id'] == 860] #.max()

,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice,fpl_id,managers,owned_by,ownwership_percent,ownership_percent
461,1,175,"R 8,180.18",daniel,2,49.5,,Moore,1.0,3,False,False,860,66.0,1,1.52,1.52
3123,2,259,"R 14,113.44",Ziggy🚀,3,40.5,,Moore,1.0,3,False,False,860,259.0,3,1.16,1.16
3550,2,259,"R 14,113.44",hiyajosephina,19,24.5,,Moore,1.0,3,False,False,860,259.0,3,1.16,1.16
3574,2,259,"R 14,113.44",Ziggy🚀,2,19.5,,Moore,1.0,3,False,False,860,259.0,3,1.16,1.16
31811,11,314,"R 15,093.13",RolandoK,1,8.0,,Moore,0.0,3,False,False,860,313.0,1,0.32,0.32


## Combine with FPL merged data


In [189]:
fpl_23_24 = pd.read_csv('../with new features/data/joint/23-24/merged_player_data.csv')
players_preds_36 = pd.read_csv('../with new features/models/preds/players_preds_36.csv')
players_preds_37 = pd.read_csv('../with new features/models/preds/players_preds_37.csv')
players_preds_38 = pd.read_csv('../with new features/models/preds/players_preds_38.csv')
# fpl_23_24[['event', 'fpl_id', 'ownership_percent']]

In [183]:
# Select only the necessary columns
ownership_lookup = fgo_df[['gameweek', 'fpl_id', 'player_name','ownership_percent']].copy()

# Ensure 'gameweek' is numeric
ownership_lookup['gameweek'] = pd.to_numeric(ownership_lookup['gameweek'])

# Drop duplicate entries to have one row per player per gameweek
ownership_lookup.drop_duplicates(subset=['gameweek', 'fpl_id', 'player_name'], inplace=True)
ownership_lookup = ownership_lookup.rename(columns={'gameweek':'event'})


# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
final_df = pd.merge(
    fpl_23_24,
    ownership_lookup,
    on=['event', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
preds_36 = pd.merge(
    players_preds_36,
    ownership_lookup,
    on=['event', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_37 = pd.merge(
    players_preds_37,
    ownership_lookup,
    on=['event', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_38 = pd.merge(
    players_preds_38,
    ownership_lookup,
    on=['event', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)


final_df['ownership_percent'] = final_df['ownership_percent'].fillna(0)
preds_36['ownership_percent'] = preds_36['ownership_percent'].fillna(0)
preds_37['ownership_percent'] = preds_37['ownership_percent'].fillna(0)
preds_38['ownership_percent'] = preds_38['ownership_percent'].fillna(0)

In [184]:
final_df.to_csv('./fantasyGo_FPL.csv')
preds_36.to_csv('./fantasyGo_preds_36.csv')
preds_37.to_csv('./fantasyGo_preds_37.csv')
preds_38.to_csv('./fantasyGo_preds_38.csv')

In [191]:
preds_36[['ownership_percent']]

,ownership_percent
0,0.00
1,0.44
2,0.87
3,0.44
4,0.00
...,...
282,0.00
283,4.37
284,0.00
285,0.00
